In [1]:
import numpy as np

np.random.seed(45)

class premSim:
    def __init__ (self, initPremium, avgPremGrowth, stdDevPremGrowth, numPaths):
        self.numPaths = numPaths
        self.premium = np.full(numPaths, initPremium, dtype=np.float64)
        self.avgPremGrowth = avgPremGrowth
        self.stdDevPremGrowth = stdDevPremGrowth
        m = self.avgPremGrowth
        v = self.stdDevPremGrowth ** 2
        self.mu = np.log(m**2 / np.sqrt(v + m**2))
        self.sigma = np.sqrt(np.log(1 + v / m**2))

    def updateParams(self):
        m = self.avgPremGrowth
        v = self.stdDevPremGrowth ** 2
        self.mu = np.log(m**2 / np.sqrt(v + m**2))
        self.sigma = np.sqrt(np.log(1 + v / m**2))

    def advanceYear(self):
        premGrowthRate = np.random.lognormal(self.mu, self.sigma, self.numPaths)
        self.premium *= premGrowthRate
        return self.premium, premGrowthRate

class polSim:
    def __init__ (self, initPols, avgPolGrowth, stdDevPolGrowth, avgElas, stdDevElas, numPaths):
        self.numPaths = numPaths
        self.pols = np.full(numPaths, initPols, dtype=np.float64)
        self.avgPolGrowth = avgPolGrowth
        self.stdDevPolGrowth = stdDevPolGrowth
        self.avgElas = avgElas
        self.stdDevElas = stdDevElas
        self.updateParams()

    def updateParams(self):
        m = self.avgPolGrowth
        v = self.stdDevPolGrowth ** 2
        self.mu = np.log(m**2 / np.sqrt(v + m**2))
        self.sigma = np.sqrt(np.log(1 + v / m**2))

    def advanceYear(self, premGrowthRate):
        polGrowthRate = np.random.lognormal(self.mu, self.sigma, self.numPaths)
        elasticity = np.random.normal(self.avgElas, self.stdDevElas, self.numPaths)
        premPctIncrease = premGrowthRate - 1
        retentionRate = np.clip(1 - (premPctIncrease * elasticity), 0.70, 0.99)
        self.pols = (self.pols * retentionRate) * polGrowthRate
        return self.pols

class claimsSim:
    def __init__ (self, numPols, avgClaimsPerPolicy, badYearClaimsPerPol, avgSmallClaims, stdDevSmallClaims, avgLargeClaims, stdDevLargeClaims, largeClaimsCap, smallClaimsDeductible, largeClaimsDeductible, smallClaimsCap, numPaths):
        self.numPols = numPols
        self.numPaths = numPaths
        self.avgClaimsPerPolicy = avgClaimsPerPolicy
        self.badYearClaimsPerPol = badYearClaimsPerPol
        self.smallClaimsCap = smallClaimsCap
        self.largeClaimsCap = largeClaimsCap
        self.smallClaimsDeductible = smallClaimsDeductible
        self.largeClaimsDeductible = largeClaimsDeductible
        self.e_s, self.v_s = 0, 0
        self.e_l, self.v_l = 0, 0
        self.updateParams(avgSmallClaims, stdDevSmallClaims, avgLargeClaims, stdDevLargeClaims)

    def updateParams(self, m_s, s_s, m_l, s_l):
        if m_s > self.smallClaimsCap:
            reductionFactor = self.smallClaimsCap / m_s
            self.e_s = self.smallClaimsCap
            self.v_s = (s_s * reductionFactor) ** 2
        else:
            self.e_s = max(m_s - self.smallClaimsDeductible, 0)
            self.v_s = s_s ** 2
        
        if m_l > self.largeClaimsCap:
            reductionFactor = self.largeClaimsCap / m_l
            self.e_l = self.largeClaimsCap
            self.v_l = (s_l * reductionFactor) ** 2
        else:
            self.e_l = max(m_l - self.largeClaimsDeductible, 0)
            self.v_l = s_l ** 2
            
    def totalClaimsCost(self):
        r = np.random.random()
        isDisaster = r < 0.05
        expClaimsPerPolicy = self.badYearClaimsPerPol if isDisaster else self.avgClaimsPerPolicy
        probSmallClaims = 0.85 if isDisaster else 0.95
        probLargeClaims = 1 - probSmallClaims
        expNumSmallClaims = (expClaimsPerPolicy * probSmallClaims) * self.numPols
        expNumLargeClaims = (expClaimsPerPolicy * probLargeClaims) * self.numPols
        expSmallClaimsTotalCost = expNumSmallClaims * self.e_s
        varSmallClaimsTotalCost = expNumSmallClaims * (self.v_s + self.e_s**2)
        expLargeClaimsTotalCost = expNumLargeClaims * self.e_l
        varLargeClaimsTotalCost = expNumLargeClaims * (self.v_l + self.e_l**2)
        totalExpClaimsCost = expSmallClaimsTotalCost + expLargeClaimsTotalCost
        totalVarClaimsCost = varSmallClaimsTotalCost + varLargeClaimsTotalCost
        pathsCost = np.random.normal(totalExpClaimsCost, np.sqrt(totalVarClaimsCost))
        return np.maximum(pathsCost, 0), r

class InsuranceModel:
    def __init__(self):
        self.probRuin = 0.0
        self.currDiscountedVaRCapital = 0.0
        self.currMeanCapital = 0.0
        self.conservativeCapital = 0.0

    def runSimulation(self):
        # Configs
        SIM_YEARS = 50
        NUM_PATHS = 100_000
        INIT_PREMIUM = 500
        INIT_NUM_POLS = 750_000
        INIT_CAPITAL = 50_000_000
        RISK_ADJUSTED_DISCOUNT_RATE = 0.97
        RECOVERY_DURATION = 2

        capitalPaths = np.full(NUM_PATHS, 100_000_000, dtype=np.float64)
        active_mask = np.ones(NUM_PATHS, dtype=bool)

        # Premium Params
        AVG_PREM_GROWTH = 1.025
        STD_PREM_GROWTH = 0.05
        POST_DISASTER_AVG_PREM_GROWTH = 1.03

        # Policy Params
        AVG_POL_GROWTH = 1.025
        STD_POL_GROWTH = 0.05
        POST_DISASTER_AVG_POL_GROWTH = 1.03

        # Claim Params
        AVG_CLAIMS_PER_POL = 0.1
        BAD_YEAR_CLAIMS_PER_POL = 0.2
        AVG_SMALL_CLAIMS = 4000
        STD_SMALL_CLAIMS = 500
        SMALL_CLAIMS_INFLATION = 1.005
        SMALL_CLAIMS_DEDUCTIBLE = 50
        SMALL_CLAIMS_CAP = 5000
        AVG_LARGE_CLAIMS = 10000
        STD_LARGE_CLAIMS = 2000
        LARGE_CLAIMS_INFLATION = 1.005
        LARGE_CLAIMS_DEDUCTIBLE = 500
        LARGE_CLAIMS_CAP = 15000

        currAvgSmallClaims = AVG_SMALL_CLAIMS
        currStdSmallClaims = STD_SMALL_CLAIMS
        currAvgLargeClaims = AVG_LARGE_CLAIMS
        currStdLargeClaims = STD_LARGE_CLAIMS
        currSmallClaimsCap = SMALL_CLAIMS_CAP
        currLargeClaimsCap = LARGE_CLAIMS_CAP
        currSmallClaimsDeductible = SMALL_CLAIMS_DEDUCTIBLE
        currLargeClaimsDeductible = LARGE_CLAIMS_DEDUCTIBLE

        # Elasticity Params
        AVG_ELAS = 0.4
        STD_ELAS = 0.125
        POST_DISASTER_AVG_ELAS = 0.25

        premModel = premSim(initPremium=INIT_PREMIUM, 
                            avgPremGrowth=AVG_PREM_GROWTH,
                            stdDevPremGrowth=STD_PREM_GROWTH,
                            numPaths = NUM_PATHS)
        
        polModel = polSim(initPols=INIT_NUM_POLS,
                          avgPolGrowth=AVG_POL_GROWTH, 
                          stdDevPolGrowth=STD_POL_GROWTH,
                          avgElas=AVG_ELAS,
                          stdDevElas=STD_ELAS,
                          numPaths=NUM_PATHS)
        
        claimsModel = claimsSim(numPols= INIT_NUM_POLS,
                                avgClaimsPerPolicy=AVG_CLAIMS_PER_POL, 
                                badYearClaimsPerPol=BAD_YEAR_CLAIMS_PER_POL,
                                avgSmallClaims=AVG_SMALL_CLAIMS, 
                                stdDevSmallClaims=STD_SMALL_CLAIMS, 
                                avgLargeClaims=AVG_LARGE_CLAIMS,
                                stdDevLargeClaims=STD_LARGE_CLAIMS,
                                smallClaimsCap = SMALL_CLAIMS_CAP,
                                largeClaimsCap = LARGE_CLAIMS_CAP,
                                smallClaimsDeductible=SMALL_CLAIMS_DEDUCTIBLE,
                                largeClaimsDeductible=LARGE_CLAIMS_DEDUCTIBLE,
                                numPaths=NUM_PATHS)

        recoveryYearsLeft = 0
        self.currVaRCapital = INIT_CAPITAL
        self.currMeanCapital = INIT_CAPITAL
        self.conservativeCapital = INIT_CAPITAL

        for year in range(1, SIM_YEARS + 1):
            currPrem, premGrowthRate = premModel.advanceYear()
            currPols = polModel.advanceYear(premGrowthRate)
            currAnnuity = currPrem * currPols
            claimsModel.numPols = currPols
            currClaimsCost, badYear = claimsModel.totalClaimsCost()

            discountFactor = RISK_ADJUSTED_DISCOUNT_RATE ** year
            netCashFLow = currAnnuity - currClaimsCost

            capitalPaths[active_mask] += (netCashFLow[active_mask]) * discountFactor

            newlyRuinedPaths = active_mask & (capitalPaths <= 0)
            active_mask[newlyRuinedPaths] = False

            self.probRuin = (NUM_PATHS - np.sum(active_mask)) / NUM_PATHS

            if badYear < 0.05:
                recoveryYearsLeft = RECOVERY_DURATION
                disaster_marker = "!!! DISASTER !!!"
            else:
                disaster_marker = ""

            if recoveryYearsLeft > 0:
                premModel.avgPremGrowth = POST_DISASTER_AVG_PREM_GROWTH
                polModel.avgElas = POST_DISASTER_AVG_ELAS
                polModel.avgPolGrowth = POST_DISASTER_AVG_POL_GROWTH
                recoveryYearsLeft -= 1
            else:
                premModel.avgPremGrowth = AVG_PREM_GROWTH
                polModel.avgElas = AVG_ELAS
                polModel.avgPolGrowth = AVG_POL_GROWTH

            premModel.updateParams()
            polModel.updateParams()

            currSmallClaimsCap *= SMALL_CLAIMS_INFLATION
            currLargeClaimsCap *= LARGE_CLAIMS_INFLATION
            currSmallClaimsDeductible *= SMALL_CLAIMS_INFLATION
            currLargeClaimsDeductible *= LARGE_CLAIMS_INFLATION

            claimsModel.smallClaimsCap = currSmallClaimsCap
            claimsModel.largeClaimsCap = currLargeClaimsCap
            claimsModel.smallClaimsDeductible = currSmallClaimsDeductible
            claimsModel.largeClaimsDeductible = currLargeClaimsDeductible

            # VaR Annuity and Claims
            VaR95_annuity = np.percentile(currAnnuity, 5)
            VaR95_claims = np.percentile(currClaimsCost, 95)

            # Discounted  and VaR Capital
            cashflow = currAnnuity - currClaimsCost
            discountedCashflow = cashflow * discountFactor
            discountedVaRCashflow = np.percentile(discountedCashflow, 5)

            # Mean Annuity and Claims
            mean_annuity = np.mean(currAnnuity)
            mean_claims = np.mean(currClaimsCost)
            
            # Discounted VaR Annuity and Claims
            discountedVaRAnnuity = VaR95_annuity * discountFactor
            discountedVaRClaims = VaR95_claims * discountFactor

            # Discounted Mean Annuity and Claims
            discountedMeanAnnuity = mean_annuity * discountFactor
            discountedMeanClaims = mean_claims * discountFactor
            
            self.currDiscountedVaRCapital += discountedVaRCashflow
            self.currMeanCapital += (discountedMeanAnnuity - discountedMeanClaims)
            self.conservativeCapital += (discountedVaRAnnuity - discountedVaRClaims)

            currAvgSmallClaims *= SMALL_CLAIMS_INFLATION
            currStdSmallClaims *= SMALL_CLAIMS_INFLATION
            currAvgLargeClaims *= LARGE_CLAIMS_INFLATION
            currStdLargeClaims *= LARGE_CLAIMS_INFLATION

            disaster_marker = "!!! DISASTER !!!" if badYear < 0.05 else ""
            print(f"Year: {year:2} | VaR Capital: {self.currDiscountedVaRCapital:>16,.2f} | Mean Capital: {self.currMeanCapital:>16,.2f} {disaster_marker}")

            claimsModel.updateParams(currAvgSmallClaims, currStdSmallClaims, currAvgLargeClaims, currStdLargeClaims)

# Usage
model = InsuranceModel()
model.runSimulation()
print(f"\nFinal Probability of Ruin: {model.probRuin:.4%}")

Year:  1 | VaR Capital:    36,503,309.92 | Mean Capital:   115,591,933.16 
Year:  2 | VaR Capital:    67,747,771.91 | Mean Capital:   187,129,930.45 
Year:  3 | VaR Capital:    95,872,090.90 | Mean Capital:   264,367,162.82 
Year:  4 | VaR Capital:   122,296,493.25 | Mean Capital:   347,205,693.99 
Year:  5 | VaR Capital:   147,918,390.36 | Mean Capital:   435,631,525.08 
Year:  6 | VaR Capital:   173,336,287.36 | Mean Capital:   529,478,033.55 
Year:  7 | VaR Capital:   199,658,105.84 | Mean Capital:   628,713,219.61 
Year:  8 | VaR Capital:   226,140,948.68 | Mean Capital:   733,161,441.95 
Year:  9 | VaR Capital:   253,811,293.82 | Mean Capital:   842,811,287.70 
Year: 10 | VaR Capital:   282,273,845.96 | Mean Capital:   957,509,564.70 
Year: 11 | VaR Capital:   312,240,902.64 | Mean Capital: 1,077,060,335.77 
Year: 12 | VaR Capital:   343,131,304.70 | Mean Capital: 1,201,446,241.79 
Year: 13 | VaR Capital:   375,554,609.09 | Mean Capital: 1,330,568,103.34 
Year: 14 | VaR Capital:  